# Using Lightgbm  Only

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import roc_auc_score
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded successfully!")

In [ ]:
print("1. Loading Data...")
# Replace paths if needed!
train = pd.read_csv('/content/drive/MyDrive/Playground-series/s6e3/train.csv')
test = pd.read_csv('/content/drive/MyDrive/Playground-series/s6e3/test.csv')

TARGET = 'Churn'
ID_COL = 'id'

In [ ]:
y = train[TARGET].map({'Yes': 1, 'No': 0})
X = train.drop(columns=[TARGET, ID_COL])
X_test = test.drop(columns=[ID_COL])

In [ ]:
print("2. Applying Feature Engineering...")
# --- THE SECRET SAUCE: FEATURE ENGINEERING ---
def apply_feature_engineering(df):
    df = df.copy()

    # TotalCharges has empty strings " " for new customers. Convert to numeric and fill with 0.
    df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0)

    # 1. Financial Features
    # Compare what they should have paid vs what they actually paid
    df['Calculated_Total'] = df['tenure'] * df['MonthlyCharges']
    df['Total_Difference'] = df['TotalCharges'] - df['Calculated_Total']

    # 2. Service Counts (More services = less likely to churn usually)
    security_cols = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport']
    # Count how many of these features they have
    df['Security_Count'] = sum((df[col] == 'Yes').astype(int) for col in security_cols)

    streaming_cols = ['StreamingTV', 'StreamingMovies']
    df['Streaming_Count'] = sum((df[col] == 'Yes').astype(int) for col in streaming_cols)

    # Total number of services they subscribe to
    df['Total_Services'] = df['Security_Count'] + df['Streaming_Count'] + (df['PhoneService'] == 'Yes').astype(int)

    # 3. Ratios
    # How much are they paying per service?
    df['Cost_Per_Service'] = df['MonthlyCharges'] / (df['Total_Services'] + 1)

    return df

In [ ]:
# Apply to both Train and Test
X = apply_feature_engineering(X)
X_test = apply_feature_engineering(X_test)

In [ ]:
print("3. Preprocessing Categoricals...")
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()

X_lgb_xgb = X.copy()
X_test_lgb_xgb = X_test.copy()
X_cat = X.copy()
X_test_cat = X_test.copy()

if len(cat_cols) > 0:
    # A. CatBoost Data (Raw Strings)
    X_cat[cat_cols] = X_cat[cat_cols].astype(str).fillna('Missing')
    X_test_cat[cat_cols] = X_test_cat[cat_cols].astype(str).fillna('Missing')

    # B. LGBM/XGB Data (Ordinal Encoded)
    encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    X_lgb_xgb[cat_cols] = X_lgb_xgb[cat_cols].astype(str).fillna('Missing')
    X_test_lgb_xgb[cat_cols] = X_test_lgb_xgb[cat_cols].astype(str).fillna('Missing')

    X_lgb_xgb[cat_cols] = encoder.fit_transform(X_lgb_xgb[cat_cols])
    X_test_lgb_xgb[cat_cols] = encoder.transform(X_test_lgb_xgb[cat_cols])

    # Clean column names for LightGBM
X_lgb_xgb.columns = ["".join (c if c.isalnum() else "_" for c in str(x)) for x in X_lgb_xgb.columns]
X_test_lgb_xgb.columns = ["".join (c if c.isalnum() else "_" for c in str(x)) for x in X_test_lgb_xgb.columns]

In [ ]:
print("4. Starting Advanced Model Training (5-Fold CV) - 3 Model Ensemble...\n")

N_FOLDS = 5
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

# Arrays to store out-of-fold predictions
lgb_oof = np.zeros(len(X))
xgb_oof = np.zeros(len(X))
cat_oof = np.zeros(len(X))

# Arrays to store test predictions
lgb_test_preds = np.zeros(len(X_test))
xgb_test_preds = np.zeros(len(X_test))
cat_test_preds = np.zeros(len(X_test))

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"========== FOLD {fold+1} ==========")
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    # LGBM/XGB Data splits (Ordinal Encoded)
    X_train_lx, X_val_lx = X_lgb_xgb.iloc[train_idx], X_lgb_xgb.iloc[val_idx]

    # CatBoost Data splits (Raw Strings)
    X_train_c, X_val_c = X_cat.iloc[train_idx], X_cat.iloc[val_idx]

    # -----------------------------------------------------------
    # MODEL 1: LIGHTGBM
    # -----------------------------------------------------------
    model_lgb = lgb.LGBMClassifier(
        n_estimators=1500, learning_rate=0.03, max_depth=6, num_leaves=31,
        subsample=0.8, colsample_bytree=0.8, random_state=42+fold, verbose=-1
    )
    model_lgb.fit(
        X_train_lx, y_train, eval_set=[(X_val_lx, y_val)],
        callbacks=[lgb.early_stopping(stopping_rounds=75, verbose=False)]
    )
    lgb_oof[val_idx] = model_lgb.predict_proba(X_val_lx)[:, 1]
    lgb_test_preds += model_lgb.predict_proba(X_test_lgb_xgb)[:, 1] / N_FOLDS

    # -----------------------------------------------------------
    # MODEL 2: XGBOOST
    # -----------------------------------------------------------
    model_xgb = xgb.XGBClassifier(
        n_estimators=1500, learning_rate=0.03, max_depth=5,
        subsample=0.8, colsample_bytree=0.8, eval_metric='auc',
        random_state=42+fold, early_stopping_rounds=75, tree_method='hist'
    )
    model_xgb.fit(X_train_lx, y_train, eval_set=[(X_val_lx, y_val)], verbose=False)
    xgb_oof[val_idx] = model_xgb.predict_proba(X_val_lx)[:, 1]
    xgb_test_preds += model_xgb.predict_proba(X_test_lgb_xgb)[:, 1] / N_FOLDS

    # -----------------------------------------------------------
    # MODEL 3: CATBOOST
    # -----------------------------------------------------------
    model_cat = CatBoostClassifier(
        iterations=1500, learning_rate=0.03, depth=6, eval_metric='AUC',
        cat_features=cat_cols, random_seed=42+fold, verbose=False
    )
    model_cat.fit(X_train_c, y_train, eval_set=(X_val_c, y_val), early_stopping_rounds=75)
    cat_oof[val_idx] = model_cat.predict_proba(X_val_c)[:, 1]
    cat_test_preds += model_cat.predict_proba(X_test_cat)[:, 1] / N_FOLDS

    # Print AUC for the current fold
    print(f"LGBM AUC: {roc_auc_score(y_val, lgb_oof[val_idx]):.5f} | "
          f"XGB AUC: {roc_auc_score(y_val, xgb_oof[val_idx]):.5f} | "
          f"CAT AUC: {roc_auc_score(y_val, cat_oof[val_idx]):.5f}")

In [ ]:
# --- OVERALL SCORES ---
print("\n======================================")
print(f"OVERALL LGBM OOF AUC: {roc_auc_score(y, lgb_oof):.5f}")
print(f"OVERALL XGB  OOF AUC: {roc_auc_score(y, xgb_oof):.5f}")
print(f"OVERALL CAT  OOF AUC: {roc_auc_score(y, cat_oof):.5f}")

# --- 5. BLEND THE MODELS ---
# We assign weights based on general tabular data performance.
# CatBoost and LGBM usually dominate, so we give them slightly higher weight.
w_lgb = 0.40
w_cat = 0.40
w_xgb = 0.20

final_oof = (lgb_oof * w_lgb) + (xgb_oof * w_xgb) + (cat_oof * w_cat)
final_test_preds = (lgb_test_preds * w_lgb) + (xgb_test_preds * w_xgb) + (cat_test_preds * w_cat)

print(f"\n🏆 BLENDED ENSEMBLE OOF AUC: {roc_auc_score(y, final_oof):.5f}")

# --- 6. CREATE SUBMISSION FILE ---
submission = pd.DataFrame({
    ID_COL: test[ID_COL],
    TARGET: final_test_preds
})

submission.to_csv('submission_ensemble.csv', index=False)
print("submission_ensemble.csv created successfully! Ready to upload to Kaggle.")

## Optimized Method

In [ ]:
from scipy.optimize import minimize

print("Finding optimal blend weights...")

# Objective function: We want to MAXIMIZE AUC, so we MINIMIZE negative AUC
def objective(weights):
    # Normalize weights so they add up to 1
    weights = weights / np.sum(weights)
    blend = (lgb_oof * weights[0]) + (xgb_oof * weights[1]) + (cat_oof * weights[2])
    return -roc_auc_score(y, blend)

# Initial guess (equal weights)
init_weights = [0.33, 0.33, 0.33]
# Bounds: weights must be between 0 and 1
bounds = [(0, 1), (0, 1), (0, 1)]

res = minimize(objective, init_weights, method='Nelder-Mead', bounds=bounds)
opt_weights = res.x / np.sum(res.x)

print(f"Optimal Weights -> LGBM: {opt_weights[0]:.4f} | XGB: {opt_weights[1]:.4f} | CAT: {opt_weights[2]:.4f}")

# Calculate new optimal OOF
optimal_oof = (lgb_oof * opt_weights[0]) + (xgb_oof * opt_weights[1]) + (cat_oof * opt_weights[2])
print(f"🚀 OPTIMIZED BLEND AUC: {roc_auc_score(y, optimal_oof):.5f}")

# Create new optimized submission
optimal_test_preds = (lgb_test_preds * opt_weights[0]) + (xgb_test_preds * opt_weights[1]) + (cat_test_preds * opt_weights[2])
submission_opt = pd.DataFrame({ID_COL: test[ID_COL], TARGET: optimal_test_preds})
submission_opt.to_csv('submission_optimized_blend.csv', index=False)
print("submission_optimized_blend.csv created!")